# 실패 분석(Failure Analysis)

이 노트북은 평가 결과 중 실패한 케이스만 따로 뽑아, 왜 실패했는지 분류하고 해석하는 과정을 다룬다. 좋은 agent 팀은 단순히 평균 점수가 올랐다고 만족하지 않고, 어떤 실패가 반복되는지 taxonomy로 정리하고, 개선 우선순위를 뽑아낸다.

## 학습 목표
- failure taxonomy가 단순 버그 메모가 아니라 개선 설계 문서라는 점을 이해한다.
- 12가지 failure type과 `critical / major / minor` severity 기준을 읽을 수 있다.
- trace와 failure metadata를 연결해 "어디서부터 이상이 시작됐는가"를 찾는 법을 익힌다.
- 면접에서 실패 분석을 어떻게 설명하면 좋은지도 정리한다.


## 개념 설명

실패 분석 노트북도 먼저 실행 환경을 확인한다. failure analysis는 이미 저장된 평가 결과와 trace 파일을 다시 읽어오는 경우가 많아서, 경로가 틀리면 "실패가 없는 것처럼" 보이는 착시가 생길 수 있다.

- **목적**: 현재 커널과 프로젝트 루트를 확인해, 평가 결과 파일과 trace 파일을 정확히 읽도록 한다.
- **핵심 로직**: 프로젝트 루트를 경로에 추가하고, `RuntimeConfig.auto_detect()`를 출력해 환경을 기록한다.
- **주요 파라미터/변수**:
  - `ROOT`: 파일 경로 해석 기준이 되는 프로젝트 루트이다.
  - `sys.executable`: 어떤 Python 환경이 연결됐는지 보여준다.
  - `RuntimeConfig.auto_detect()`: 현재 runtime 설정을 요약한다.

이런 시작 셀은 지루해 보여도, 아티팩트 기반 분석에서는 특히 중요하다. 이전 실험 결과와 현재 경로가 섞이면 엉뚱한 진단을 내리기 쉽다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## Agent 시스템의 failure mode

agent의 실패는 단순히 "답이 틀렸다" 한 줄로 끝나지 않는다. retrieval이 틀렸는지, query classification이 잘못됐는지, planner가 단계를 빠뜨렸는지, synthesis가 근거 없이 내용을 부풀렸는지에 따라 고쳐야 할 지점이 완전히 달라진다. 그래서 failure taxonomy를 만든다.

실무에서 failure taxonomy는 버그 리포트와 다르다. 버그 리포트가 개별 현상을 기록하는 문서라면, taxonomy는 **여러 실패를 공통 원인 범주로 묶어 우선순위를 정하는 체계**다.

또한 severity도 중요하다.
- `critical`: 잘못된 확신, 안전성 훼손, 신뢰 붕괴로 이어질 수 있는 실패
- `major`: 품질 저하가 뚜렷하고 사용자 경험에 큰 영향을 주는 실패
- `minor`: 결과는 나쁘지만 즉시 위험하진 않은 개선 후보

💡 면접 포인트: "failure taxonomy를 만들면 모델의 문제를 기능 팀이 바로 행동 가능한 개선 항목으로 바꿀 수 있다"고 설명하면 좋다.


## 구현

이 셀은 분석의 입력이 되는 evaluation results를 불러온다. 이미 저장된 `eval_results.json`이 있으면 재사용하고, 없으면 작은 evaluation run을 다시 실행한다. 이렇게 해두면 노트북이 단독으로도 실행 가능하고, 이전 결과가 있을 때는 불필요한 재계산을 줄일 수 있다.

- **목적**: failure analysis의 원본 데이터프레임을 준비한다.
- **핵심 로직**: `get_paths()`로 경로를 가져오고, `results_path.exists()` 여부에 따라 저장된 결과를 읽거나 `run_evaluation_suite()`를 실행한다.
- **주요 파라미터/변수**:
  - `paths`: eval, traces, reports 디렉터리 경로를 담은 설정 객체이다.
  - `results_path`: 평가 결과 JSON 파일 경로이다.
  - `results`: 이후 모든 failure 분석의 출발점이 되는 데이터프레임이다.

아래 코드에서:
- `if results_path.exists()`: 이미 있는 결과를 재활용한다.
- `else: results, _ = run_evaluation_suite(...)`: 결과가 없을 때도 노트북이 끊기지 않게 만든다.

출력 표에서는 질문 ID, 시스템 이름, 상태, 주요 metric이 어떤 식으로 들어 있는지 먼저 감을 잡아두면 뒤 셀이 훨씬 읽기 쉬워진다.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.config import get_paths
from src.evaluator import attach_failure_improvements, extract_failure_cases, run_evaluation_suite
from src.failure_analyzer import analyze_failures, generate_failure_report, taxonomy_frame
from src.trace_debug import display_trace
from src.utils import read_json

paths = get_paths()
results_path = paths.eval_dir / 'eval_results.json'
if results_path.exists():
    results = pd.DataFrame(read_json(results_path))
else:
    results, _ = run_evaluation_suite(repeats=2, persist_outputs=True)

results.head(5)

## 실패 taxonomy

이 셀은 failure taxonomy 자체를 표로 보여준다. 단순히 이름만 나열하는 것이 아니라, 각 failure type이 어느 stage에서 발생하는지, severity가 어느 수준인지, 어떤 완화 방안(mitigation)이 연결되는지 함께 읽는 것이 중요하다.

이 프로젝트의 taxonomy는 예를 들어 다음과 같은 유형을 포함한다.
- `retrieval_miss`: 맞는 문서를 top-k 안에 못 가져온 경우
- `retrieval_noise`: 문서는 가져왔지만 노이즈가 너무 많은 경우
- `query_misclassification`: 질문 라우팅이 잘못된 경우
- `bad_plan`, `missing_decomposition`: plan이 부실하거나 필요한 단계가 빠진 경우
- `tool_execution_error`: 도구 실행 자체가 실패한 경우
- `ungrounded_synthesis`, `citation_mismatch`, `incomplete_synthesis`: 답변 합성 품질 문제
- `insufficient_evidence_not_detected`, `over_abstention`: verifier/fallback 판단 문제

- **목적**: failure type, severity, stage를 구조화된 표로 확인한다.
- **핵심 로직**: `taxonomy_frame()`을 정렬해 표로 만들고, severity별 색을 입혀 한눈에 읽기 쉽게 만든다.
- **주요 파라미터/변수**:
  - `severity_colors`: critical/major/minor에 대응하는 배경색 매핑이다.
  - `taxonomy`: 정렬된 taxonomy 데이터프레임이다.

결과를 볼 때는 색이 진한 것만 보는 것이 아니라, 어떤 stage에 failure가 몰리는지 함께 보자. 같은 major라도 retrieval 단계 문제와 verification 단계 문제는 대응 전략이 다르다.


In [ ]:
taxonomy = taxonomy_frame().sort_values(['severity', 'stage', 'failure_type']).reset_index(drop=True)
severity_colors = {'critical': '#FEE2E2', 'major': '#FEF3C7', 'minor': '#DBEAFE'}

def color_row(row):
    color = severity_colors.get(row['severity'], '#FFFFFF')
    return [f'background-color: {color}' for _ in row]

taxonomy.style.apply(color_row, axis=1)

## 실패 케이스 추출(Extract failure cases)

전체 평가 결과에는 성공 케이스와 실패 케이스가 섞여 있다. 개선 작업을 하려면 먼저 실패 행만 분리하고, 각 실패에 대해 "다음에 무엇을 고치면 좋은가"라는 개선 아이디어를 붙이는 것이 유용하다. 이 셀은 바로 그 작업을 한다.

- **목적**: 원본 evaluation results에서 실패 행만 추출하고, 개선 힌트를 붙인다.
- **핵심 로직**: `extract_failure_cases(results)`가 실패 케이스만 골라내고, `attach_failure_improvements(...)`가 각 failure type에 대응하는 개선 아이디어를 덧붙인다.
- **주요 파라미터/변수**:
  - `failures`: 실패 케이스만 담은 데이터프레임이다.
  - `failure_type`: 어떤 범주의 실패인지 나타낸다.
  - `improvement_idea`: 다음 실험에서 시도해볼 개선 방향이다.

이 표를 읽을 때는 한 건 한 건보다 반복 패턴을 보는 것이 중요하다. 같은 failure type이 여러 질문에서 반복된다면, 개별 질문 수정이 아니라 시스템 수준의 개선이 필요하다는 뜻이다.


In [ ]:
failures = attach_failure_improvements(extract_failure_cases(results))
failures[['system', 'question_id', 'expected_question_type', 'failure_type', 'improvement_idea']].head(12)

## trace 점검(Trace inspection)

분류만으로는 부족하다. 정말로 어디서 문제가 시작됐는지 확인하려면 개별 실패 케이스의 trace를 열어봐야 한다. 이 셀은 전체 failure distribution을 집계하고, markdown 리포트를 생성한 뒤, 대표적인 agent failure 3건의 trace를 직접 보여준다.

- **목적**: failure type 통계와 개별 실행 흔적을 연결해 읽는다.
- **핵심 로직**: `analyze_failures(failures)`로 집계를 만들고, `generate_failure_report(...)`로 보고서를 저장한 뒤, 상위 몇 개 실패 케이스의 trace 파일을 열어 `display_trace()`로 시각화한다.
- **주요 파라미터/변수**:
  - `analysis`: failure distribution, stage distribution, severity distribution 등을 담은 분석 결과이다.
  - `report_path`: 자동 생성되는 실패 분석 보고서 경로이다.
  - `trace_previews`: 어떤 질문의 어떤 trace를 볼지 정리한 미리보기 표이다.

실제 디버깅에서는 여기서부터가 중요하다. 예를 들어 failure type이 `bad_plan`이어도, trace를 열어보면 classifier가 잘못 분류해서 plan이 엇나간 것일 수 있다. taxonomy는 출발점이고, trace는 원인 확인 도구다.


In [ ]:
from pathlib import Path

analysis = analyze_failures(failures)
report_path = paths.reports_dir / 'failure_report.md'
generate_failure_report(analysis, report_path)

agent_failures = failures[failures['system'] == 'agent_workflow'].head(3)
trace_previews = []
for _, row in agent_failures.iterrows():
    trace_path = paths.traces_dir / f"{row['question_id']}_run{int(row['run_id'])}.json"
    trace = read_json(trace_path)['trace'] if trace_path.exists() else []
    trace_previews.append(
        {
            'question_id': row['question_id'],
            'failure_type': row['failure_type'],
            'trace_path': str(trace_path),
            'trace_steps': len(trace),
        }
    )

trace_preview_frame = pd.DataFrame(trace_previews)
display(trace_preview_frame)
for preview in trace_previews:
    print(f"Trace for {preview['question_id']} ({preview['failure_type']})")
    display_trace(read_json(Path(preview['trace_path']))['trace'])

## 실험

이 셀은 실패를 집계 수준에서 다시 본다. stage별 실패 수는 어디에서 병목이 생기는지 보여주고, severity 분포는 지금 가장 위험한 문제가 무엇인지 알려준다. `top_improvement_actions` 표는 당장 무엇부터 시도할지 우선순위를 정하는 데 도움을 준다.

- **목적**: 실패를 stage와 severity 축으로 요약해 개선 우선순위를 잡는다.
- **핵심 로직**: `analysis`에서 분포를 꺼내 막대그래프와 파이차트로 그린 뒤, 개선 액션 표를 함께 표시한다.
- **주요 파라미터/변수**:
  - `stage_distribution`: 어느 workflow stage에서 실패가 많이 발생했는지 보여준다.
  - `severity_distribution`: critical/major/minor 비율이다.
  - `improvement_actions`: 추천 개선 항목 표이다.

결과를 읽을 때는 단순히 가장 큰 막대만 보지 말고, critical이 어느 stage에 몰려 있는지 먼저 확인하자. minor가 많아도 critical 한두 개가 더 시급할 수 있다.


In [ ]:
stage_distribution = pd.Series(analysis['stage_distribution']).sort_values(ascending=False)
severity_distribution = pd.Series(analysis['severity_distribution']).sort_values(ascending=False)
improvement_actions = pd.DataFrame(analysis['top_improvement_actions'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
stage_distribution.plot(kind='bar', color='#4C78A8', ax=axes[0], title='Failures by Stage')
axes[0].set_xlabel('stage')
axes[0].set_ylabel('count')
severity_distribution.plot(kind='pie', autopct='%1.0f%%', ax=axes[1], title='Failure Severity Mix')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

display(improvement_actions)

## 결과 해석

이 마지막 요약 셀은 failure analysis 결과를 면접이나 리뷰 자리에서 한 문장으로 정리하기 쉽게 만들어준다. 전체 실패 수, failure type 다양성, 생성된 리포트 경로, 가장 우선순위가 높은 개선 행동을 한 번에 보여준다.

- **목적**: 실패 분석 결과를 실행 가능한 요약으로 압축한다.
- **핵심 로직**: `analysis`와 `improvement_actions`에서 핵심 수치만 뽑아 `Series`로 정리한다.
- **주요 파라미터/변수**:
  - `total_failures`: 전체 실패 인스턴스 수이다.
  - `unique_failure_types`: 이번 평가에서 관찰된 실패 유형 가짓수이다.
  - `top_improvement_action`: 지금 가장 먼저 시도할 개선 방안이다.

이 결과를 해석할 때는 "실패가 많다/적다"보다 "어떤 실패가 반복되는가"에 집중하자. failure analysis의 목적은 시스템을 비판하는 것이 아니라, 다음 실험을 더 똑똑하게 설계하는 것이다.

면접에서는 "평가 점수만 보지 않고, failure taxonomy와 trace를 연결해 개선 backlog를 만들었다"고 설명하면 강한 인상을 줄 수 있다.


In [ ]:
pd.Series({
    'total_failures': int(analysis['total_failure_instances']),
    'unique_failure_types': len(analysis['failure_distribution']),
    'report_path': str(report_path),
    'top_improvement_action': improvement_actions.iloc[0]['mitigation'] if not improvement_actions.empty else 'None',
})

## 핵심 정리

이 노트북을 통해 failure analysis는 단순한 에러 수집이 아니라, agent 개선 전략을 구조화하는 작업이라는 점을 확인했다. taxonomy는 실패를 공통 원인으로 묶고, severity는 무엇부터 고쳐야 하는지 우선순위를 정해주며, trace는 실제 원인 지점을 확인하게 해준다.

또한 좋은 실패 분석은 "틀렸다"에서 끝나지 않고, "다음에 무엇을 바꿀 것인가"까지 연결된다. 그래서 improvement action이 taxonomy와 함께 존재해야 한다.

💡 면접 포인트: "평가 결과를 보고 끝내지 않고, failure taxonomy·severity·trace를 연결해 개선 계획으로 전환했다"고 말하면, 단순 모델 실험이 아니라 운영 가능한 ML/agent 사고방식을 보여줄 수 있다.
